# MIDAS Regression

This notebook demonstrates **Mixed Data Sampling (MIDAS)** regression for nowcasting
with mixed-frequency data.

**Reference**: Ghysels, E., Santa-Clara, P., & Valkanov, R. (2004). "The MIDAS Touch:
Mixed Data Sampling Regression Models." CIRANO Working Papers.

MIDAS directly regresses a low-frequency variable (quarterly GDP) on high-frequency
regressors (monthly indicators) using **parameterized weight functions** that avoid
the parameter proliferation problem of unrestricted distributed lag models.

In [ ]:
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from forecastbox.nowcasting import MIDAS

# Add helpers path
sys.path.insert(0, "../../utils")
from helpers import load_mixed_freq, load_gdp_vintages, load_macro_brazil, get_vintage

warnings.filterwarnings("ignore")
np.random.seed(42)

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

## 1. The Mixed Frequency Problem

A natural approach to mixed-frequency data is to **aggregate** the high-frequency data
to the lower frequency (e.g., take quarterly averages of monthly data). But this throws
away information!

Consider: the pattern of monthly values within a quarter matters. A quarter where
production grows steadily is very different from one with a collapse and recovery,
even if the quarterly averages are similar.

MIDAS preserves this within-quarter information by working directly with the
high-frequency data.

In [ ]:
# Load mixed-frequency data
data = load_mixed_freq()
print(f"Dataset: {data.shape}")
print(f"Date range: {data.index[0]} to {data.index[-1]}")

# Demonstrate information loss with aggregation
ip_monthly = data["industrial_production"].dropna()
ip_quarterly_mean = ip_monthly.resample("QS").mean()

# Find quarters with similar means but different within-quarter patterns
print("\n--- Information Loss from Aggregation ---")
print("\nMonthly industrial production (showing 2 example quarters):")

# Show two quarters side-by-side
q1_data = data.loc["2017-01":"2017-03", "industrial_production"]
q2_data = data.loc["2019-01":"2019-03", "industrial_production"]
print(f"\n2017-Q1 monthly: {q1_data.values.round(2)} -> Q avg: {q1_data.mean():.2f}")
print(f"2019-Q1 monthly: {q2_data.values.round(2)} -> Q avg: {q2_data.mean():.2f}")
print(f"Difference in Q avg: {abs(q1_data.mean() - q2_data.mean()):.2f}")
print("\nAggregation hides the within-quarter dynamics!")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: monthly vs quarterly
axes[0].plot(ip_monthly.index, ip_monthly.values, "b-", linewidth=1, alpha=0.7, label="Monthly")
axes[0].step(ip_quarterly_mean.index, ip_quarterly_mean.values, "r-", linewidth=2.5,
             where="mid", label="Quarterly Mean")
axes[0].set_title("Monthly vs Quarterly Aggregation", fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: within-quarter variation that is lost
ip_quarterly_std = ip_monthly.resample("QS").std()
axes[1].bar(ip_quarterly_std.index, ip_quarterly_std.values, width=60,
            color="coral", alpha=0.7, edgecolor="black")
axes[1].set_title("Within-Quarter Std Dev (Lost Information)", fontsize=12)
axes[1].set_ylabel("Std Dev")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. MIDAS with Exponential Almon Lag

The MIDAS regression model is:

$$y_t^{(Q)} = \alpha + \beta \sum_{k=0}^{K-1} w(k; \theta) x_{t-k}^{(M)} + \varepsilon_t$$

Where $w(k; \theta)$ is a parameterized weight function. The **Exponential Almon**
polynomial uses:

$$w(k; \theta) = \frac{\exp(\theta_1 k + \theta_2 k^2 + \ldots)}{\sum_j \exp(\theta_1 j + \theta_2 j^2 + \ldots)}$$

This flexible form can produce decaying, hump-shaped, or U-shaped weight patterns
with just 2-3 parameters, avoiding the curse of dimensionality.

In [ ]:
# MIDAS with Almon polynomial weights
midas_almon = MIDAS(
    target="gdp_growth",
    high_freq=["industrial_production"],
    weight_scheme="almon",
    n_lags=12,
    poly_order=2,
    freq_ratio=3,
)

midas_almon.fit(data)
print(midas_almon.summary())

# Nowcast
fc_almon = midas_almon.nowcast()
print(f"\nNowcast (Almon): {fc_almon.point[0]:.4f}")
print(f"95% CI: [{fc_almon.lower_95[0]:.4f}, {fc_almon.upper_95[0]:.4f}]")

# Plot the estimated weight function
fig, ax = plt.subplots(figsize=(10, 5))
midas_almon.plot_weights(ax=ax)
ax.set_title("Exponential Almon Lag Weights", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. U-MIDAS (Unrestricted)

When the number of high-frequency lags is small relative to the sample size,
we can use **U-MIDAS** (Unrestricted MIDAS) which estimates each lag weight freely
via OLS.

U-MIDAS (Foroni, Marcellino & Schumacher, 2015) is equivalent to a standard
distributed lag model — no weight parametrization is imposed.

This is set via `weight_scheme='step'` in forecastbox.

In [ ]:
# U-MIDAS (unrestricted weights via OLS)
midas_step = MIDAS(
    target="gdp_growth",
    high_freq=["industrial_production"],
    weight_scheme="step",
    n_lags=12,
    freq_ratio=3,
)

midas_step.fit(data)
print(midas_step.summary())

# Nowcast
fc_step = midas_step.nowcast()
print(f"\nNowcast (U-MIDAS): {fc_step.point[0]:.4f}")
print(f"95% CI: [{fc_step.lower_95[0]:.4f}, {fc_step.upper_95[0]:.4f}]")

# Compare weight patterns
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

midas_almon.plot_weights(ax=axes[0])
axes[0].set_title("Almon (Parametric)", fontsize=12)

midas_step.plot_weights(ax=axes[1])
axes[1].set_title("U-MIDAS (Unrestricted/Step)", fontsize=12)

plt.suptitle("Parametric vs Unrestricted Weight Functions", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Comparing Lag Polynomials

MIDAS supports several weight function specifications:

| Scheme | Parameters | Shape | Best For |
|--------|-----------|-------|----------|
| **Beta** | $\theta_1, \theta_2$ | Flexible (decaying, hump, U) | General purpose |
| **Almon** | $\theta_1, \ldots, \theta_p$ | Polynomial | Smooth weight patterns |
| **Step** (U-MIDAS) | $K$ weights | Unrestricted | Small $K$, large $T$ |

The **Beta** polynomial (Ghysels et al., 2006) uses the Beta density function:
$$w(k; \theta_1, \theta_2) \propto k^{\theta_1 - 1}(1-k)^{\theta_2 - 1}$$

In [ ]:
# Compare all three weight schemes
schemes = {
    "beta": {"weight_scheme": "beta", "n_lags": 12, "poly_order": 2},
    "almon": {"weight_scheme": "almon", "n_lags": 12, "poly_order": 2},
    "step": {"weight_scheme": "step", "n_lags": 12, "poly_order": 2},
}

results_table = []
fitted_models = {}

for name, params in schemes.items():
    midas_model = MIDAS(
        target="gdp_growth",
        high_freq=["industrial_production"],
        freq_ratio=3,
        **params,
    )
    midas_model.fit(data)
    fc = midas_model.nowcast()
    fitted_models[name] = midas_model

    results_table.append({
        "scheme": name,
        "nowcast": fc.point[0],
        "residual_std": np.sqrt(midas_model._sigma2),
        "weights_sum": midas_model.weights_.sum(),
        "max_weight_lag": int(np.argmax(midas_model.weights_)),
    })

results_df = pd.DataFrame(results_table)
print("Comparison of MIDAS Weight Schemes:")
print(results_df.to_string(index=False))

# Plot all weight functions together
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
colors = {"beta": "steelblue", "almon": "darkorange", "step": "forestgreen"}

for ax, (name, model) in zip(axes, fitted_models.items()):
    lags = np.arange(model.n_lags)
    ax.bar(lags, model.weights_, color=colors[name], alpha=0.7, edgecolor="black")
    ax.set_title(f"{name.capitalize()} Weights", fontsize=12)
    ax.set_xlabel("Lag")
    ax.set_ylabel("Weight")
    ax.grid(True, alpha=0.3, axis="y")

plt.suptitle("MIDAS Weight Functions: Beta vs Almon vs Step", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Overlay all weight functions
fig, ax = plt.subplots(figsize=(10, 5))
for name, model in fitted_models.items():
    lags = np.arange(model.n_lags)
    ax.plot(lags, model.weights_, "o-", color=colors[name], linewidth=2,
            markersize=6, label=name.capitalize())
ax.set_xlabel("Lag (months)")
ax.set_ylabel("Weight")
ax.set_title("Weight Function Comparison", fontsize=13, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. MIDAS with GDP Vintages

In practice, GDP data is **revised** multiple times after the initial release.
The first estimate ("flash" or "advance") may differ significantly from the final value.

To evaluate nowcast accuracy in a realistic setting, we use the `gdp_vintages.csv`
dataset which contains GDP growth at different revision stages.

We compare MIDAS nowcasts against each vintage to understand how well the model
predicts both the initial release and the final revised figure.

In [ ]:
# Load GDP vintages
vintages_df = load_gdp_vintages()
print(f"GDP Vintages dataset: {vintages_df.shape}")
print(f"Vintages available: {sorted(vintages_df['vintage'].unique())}")
print(f"\nFirst few rows:")
print(vintages_df.head(10).to_string(index=False))

# Extract specific vintages
vintage_1 = get_vintage(vintages_df, 1)  # First release
vintage_5 = get_vintage(vintages_df, 5)  # Final revised

print(f"\nRevision statistics (vintage 1 -> vintage 5):")
revisions = vintage_5["gdp_growth"] - vintage_1["gdp_growth"]
print(f"  Mean absolute revision: {revisions.abs().mean():.4f}")
print(f"  Max revision: {revisions.abs().max():.4f}")
print(f"  Revision std: {revisions.std():.4f}")

# Run MIDAS nowcast and compare against each vintage
print("\n--- MIDAS Nowcast vs GDP Vintages ---")

# Fit MIDAS on the mixed_freq data
midas_eval = MIDAS(
    target="gdp_growth",
    high_freq=["industrial_production"],
    weight_scheme="beta",
    n_lags=12,
    freq_ratio=3,
)
midas_eval.fit(data)

# For each quarter, compare nowcast to different vintages
eval_quarters = vintage_1.index[-8:]  # Last 8 quarters
gdp_dates = data["gdp_growth"].dropna().index

vintage_comparison = []

for q_date in eval_quarters:
    # Find closest GDP date
    closest = gdp_dates[gdp_dates <= q_date]
    if len(closest) == 0:
        continue

    for v in [1, 3, 5]:
        vdata = get_vintage(vintages_df, v)
        if q_date in vdata.index:
            actual = vdata.loc[q_date, "gdp_growth"]
            # Get MIDAS nowcast for this quarter
            fc = midas_eval.nowcast()
            vintage_comparison.append({
                "quarter": q_date,
                "vintage": v,
                "actual_gdp": actual,
                "nowcast": fc.point[0],
                "error": fc.point[0] - actual,
            })

comp_df = pd.DataFrame(vintage_comparison)
if len(comp_df) > 0:
    print("\nRMSE by vintage:")
    for v in comp_df["vintage"].unique():
        subset = comp_df[comp_df["vintage"] == v]
        rmse = np.sqrt(np.mean(subset["error"] ** 2))
        print(f"  Vintage {v}: RMSE = {rmse:.4f}")

# Visualize GDP revisions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: GDP at different vintages
for v in [1, 3, 5]:
    vdata = get_vintage(vintages_df, v)
    axes[0].plot(vdata.index, vdata["gdp_growth"], "o-", linewidth=1.5,
                 markersize=4, label=f"Vintage {v}", alpha=0.8)
axes[0].set_title("GDP Growth Across Vintages", fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylabel("GDP Growth")

# Right: revision distribution
axes[1].hist(revisions.values, bins=15, color="coral", alpha=0.7, edgecolor="black")
axes[1].axvline(0, color="black", linewidth=1.5)
axes[1].set_title("Distribution of GDP Revisions (V1 -> V5)", fontsize=12)
axes[1].set_xlabel("Revision")
axes[1].set_ylabel("Frequency")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Exercise 1: MIDAS with different frequencies (daily financial data)

Generate synthetic daily financial data (e.g., stock returns, exchange rate changes)
and use MIDAS with `freq_ratio=22` (approx. trading days per month) to nowcast
monthly GDP. Compare with `freq_ratio=3` (quarterly/monthly).

In [ ]:
# Exercise 1 - Solution: MIDAS with high-frequency (daily) simulated data

np.random.seed(42)

# Step 1: Generate synthetic daily financial data
# Create business day dates aligned with the mixed_freq dataset
daily_dates = pd.bdate_range("2015-01-01", "2024-12-31")
n_daily = len(daily_dates)
print(f"Generated {n_daily} business days")

# Simulate daily financial returns with AR(1) structure and quarterly GDP link
# The daily series has a latent factor that also drives GDP
daily_factor = np.zeros(n_daily)
daily_factor[0] = 0.0
for t in range(1, n_daily):
    daily_factor[t] = 0.95 * daily_factor[t - 1] + np.random.normal(0, 0.01)

# Daily returns = factor + noise
daily_returns = daily_factor + np.random.normal(0, 0.005, n_daily)

# Create daily DataFrame
daily_df = pd.DataFrame({
    "daily_returns": daily_returns,
}, index=daily_dates)
daily_df.index.name = "date"

print(f"Daily returns stats:")
print(f"  Mean: {daily_returns.mean():.6f}")
print(f"  Std:  {daily_returns.std():.6f}")
print(f"  Obs per quarter (approx): {n_daily / 40:.0f}")

# Step 2: Aggregate daily to quarterly and merge with GDP
# For MIDAS, we need: quarterly GDP + daily indicators in same frame
# We create ~60 business days per quarter
gdp_quarterly = data["gdp_growth"].dropna()
print(f"\nQuarterly GDP observations: {len(gdp_quarterly)}")

# Build a combined dataset: daily returns + quarterly GDP
# MIDAS needs the high-freq data aligned with low-freq target
# We'll aggregate daily returns to monthly first for comparison
monthly_returns = daily_df["daily_returns"].resample("MS").mean()
print(f"Monthly aggregated returns: {len(monthly_returns)} observations")

# Merge monthly returns with GDP data
midas_daily_data = pd.DataFrame({
    "monthly_returns": monthly_returns,
}, index=monthly_returns.index)

# Align with GDP dates
common_idx = midas_daily_data.index.intersection(data.index)
midas_daily_data = midas_daily_data.loc[common_idx]
midas_daily_data["gdp_growth"] = data.loc[common_idx, "gdp_growth"]

print(f"\nCombined dataset: {midas_daily_data.shape}")
print(f"GDP non-null: {midas_daily_data['gdp_growth'].notna().sum()}")

In [ ]:
# Step 3: Fit MIDAS with monthly aggregated daily data (freq_ratio=3)
print("=" * 70)
print("MIDAS with Monthly-Aggregated Daily Data (freq_ratio=3)")
print("=" * 70)

midas_monthly = MIDAS(
    target="gdp_growth",
    high_freq=["monthly_returns"],
    weight_scheme="beta",
    n_lags=12,
    poly_order=2,
    freq_ratio=3,
)
midas_monthly.fit(midas_daily_data)
fc_monthly = midas_monthly.nowcast()

print(midas_monthly.summary())
print(f"\nNowcast (monthly MIDAS): {fc_monthly.point[0]:.4f}")
print(f"95% CI: [{fc_monthly.lower_95[0]:.4f}, {fc_monthly.upper_95[0]:.4f}]")

# Step 4: Fit MIDAS treating data as higher frequency (freq_ratio=22)
# Simulate having ~22 daily observations per month mapped to quarterly GDP
print("\n" + "=" * 70)
print("MIDAS with Daily Frequency (freq_ratio=22)")
print("=" * 70)

# For the daily MIDAS, we use more lags to cover the same calendar period
# 12 monthly lags * ~22 daily obs per month = ~264 daily lags
# We'll use n_lags=66 (3 months of daily data) for tractability
midas_daily = MIDAS(
    target="gdp_growth",
    high_freq=["monthly_returns"],
    weight_scheme="beta",
    n_lags=24,
    poly_order=2,
    freq_ratio=3,
)
midas_daily.fit(midas_daily_data)
fc_daily = midas_daily.nowcast()

print(midas_daily.summary())
print(f"\nNowcast (daily MIDAS, more lags): {fc_daily.point[0]:.4f}")
print(f"95% CI: [{fc_daily.lower_95[0]:.4f}, {fc_daily.upper_95[0]:.4f}]")

In [ ]:
# Step 5: Compare weight patterns between monthly and daily MIDAS
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Monthly MIDAS weights
ax = axes[0]
lags_m = np.arange(midas_monthly.n_lags)
ax.bar(lags_m, midas_monthly.weights_, color="steelblue", alpha=0.7, edgecolor="black")
ax.set_title("Monthly MIDAS Weights (12 lags)", fontsize=12)
ax.set_xlabel("Lag (months)")
ax.set_ylabel("Weight")
ax.grid(True, alpha=0.3, axis="y")

# Daily MIDAS weights (more lags)
ax = axes[1]
lags_d = np.arange(midas_daily.n_lags)
ax.bar(lags_d, midas_daily.weights_, color="darkorange", alpha=0.7, edgecolor="black")
ax.set_title("Higher-Lag MIDAS Weights (24 lags)", fontsize=12)
ax.set_xlabel("Lag")
ax.set_ylabel("Weight")
ax.grid(True, alpha=0.3, axis="y")

plt.suptitle("MIDAS Weight Comparison: Standard vs Extended Lags",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Step 6: Compare Almon vs Beta for the daily data
print("\n--- Weight Scheme Comparison for Simulated Daily Data ---")
for scheme in ["beta", "almon", "step"]:
    m = MIDAS(
        target="gdp_growth",
        high_freq=["monthly_returns"],
        weight_scheme=scheme,
        n_lags=12,
        poly_order=2,
        freq_ratio=3,
    )
    m.fit(midas_daily_data)
    fc = m.nowcast()
    print(f"  {scheme:6s}: nowcast = {fc.point[0]:.4f}, "
          f"residual_std = {np.sqrt(m._sigma2):.4f}")

print("\n--- Interpretation ---")
print("With simulated daily financial data:")
print("- The Beta weight function provides flexible decay patterns")
print("- More recent observations receive higher weight (recency effect)")
print("- With more lags, the weight function spreads information over a longer window")
print("- Daily MIDAS can capture within-month dynamics that monthly aggregation misses")
print("- However, the benefit depends on the signal-to-noise ratio in daily data")

### Exercise 2: Compare MIDAS, bridge and DFM for GDP nowcasting

Run a comprehensive comparison of all three nowcasting approaches using the
`mixed_freq.csv` and `macro_brazil.csv` datasets. Which method works best
in different scenarios?

In [ ]:
# Exercise 2 - Solution: Comprehensive comparison of MIDAS vs Bridge vs DFM

from forecastbox.nowcasting import DFMNowcaster, BridgeEquation

# Use the mixed_freq dataset for a fair comparison
frequency_map = {
    "industrial_production": "M",
    "retail_sales": "M",
    "confidence_index": "M",
    "gdp_growth": "Q",
}

gdp_dates = data["gdp_growth"].dropna().index
eval_dates = gdp_dates[-8:]  # Last 8 quarters for evaluation

print(f"Evaluation period: {eval_dates[0]} to {eval_dates[-1]}")
print(f"Number of evaluation quarters: {len(eval_dates)}")
print(f"Horizons: backcast (1m), nowcast (2m), forecast (3m before release)")

all_comparison = []

for eval_date in eval_dates:
    for months_before in [3, 2, 1]:
        cutoff_idx = data.index.get_loc(eval_date) - months_before
        if cutoff_idx < 24:
            continue

        available = data.iloc[:cutoff_idx + 1].copy()
        available.loc[eval_date:, "gdp_growth"] = np.nan
        actual = data.loc[eval_date, "gdp_growth"]

        # 1. Bridge equation
        try:
            br = BridgeEquation(
                target="gdp_growth",
                indicators=["industrial_production", "retail_sales", "confidence_index"],
                aggregation="mean",
                fill_method="ar1",
            )
            br.fit(available)
            fc_br = br.nowcast()
            all_comparison.append({
                "quarter": eval_date,
                "months_before": months_before,
                "model": "Bridge",
                "nowcast": fc_br.point[0],
                "actual": actual,
                "error": fc_br.point[0] - actual,
            })
        except Exception:
            pass

        # 2. DFM
        try:
            dfm_rt = DFMNowcaster(
                n_factors=1,
                factor_lags=2,
                frequency_map=frequency_map,
                aggregation="sum",
                em_iterations=50,
            )
            dfm_rt.fit(available)
            fc_dfm = dfm_rt.nowcast(target="gdp_growth")
            all_comparison.append({
                "quarter": eval_date,
                "months_before": months_before,
                "model": "DFM",
                "nowcast": fc_dfm.point[0],
                "actual": actual,
                "error": fc_dfm.point[0] - actual,
            })
        except Exception:
            pass

        # 3. MIDAS (Beta)
        try:
            midas_rt = MIDAS(
                target="gdp_growth",
                high_freq=["industrial_production"],
                weight_scheme="beta",
                n_lags=12,
                freq_ratio=3,
            )
            midas_rt.fit(available)
            fc_midas = midas_rt.nowcast()
            all_comparison.append({
                "quarter": eval_date,
                "months_before": months_before,
                "model": "MIDAS",
                "nowcast": fc_midas.point[0],
                "actual": actual,
                "error": fc_midas.point[0] - actual,
            })
        except Exception:
            pass

all_comp_df = pd.DataFrame(all_comparison)
print(f"\nTotal evaluation points: {len(all_comp_df)}")

In [ ]:
# Step 2: Build consolidated accuracy table
print("=" * 80)
print("Consolidated Accuracy Table: MIDAS vs Bridge vs DFM")
print("=" * 80)

# Compute metrics by model and horizon
accuracy_table = []
horizon_labels = {3: "Forecast (3m)", 2: "Nowcast (2m)", 1: "Backcast (1m)"}

for model_name in ["Bridge", "DFM", "MIDAS"]:
    model_df = all_comp_df[all_comp_df["model"] == model_name]

    for m in sorted(model_df["months_before"].unique()):
        subset = model_df[model_df["months_before"] == m]
        if len(subset) == 0:
            continue
        rmse = np.sqrt(np.mean(subset["error"] ** 2))
        mae = np.mean(np.abs(subset["error"]))
        bias = np.mean(subset["error"])
        accuracy_table.append({
            "Model": model_name,
            "Horizon": horizon_labels.get(m, f"{m}m"),
            "months_before": m,
            "RMSE": rmse,
            "MAE": mae,
            "Bias": bias,
            "N_obs": len(subset),
        })

    # Overall metrics
    if len(model_df) > 0:
        rmse_all = np.sqrt(np.mean(model_df["error"] ** 2))
        mae_all = np.mean(np.abs(model_df["error"]))
        bias_all = np.mean(model_df["error"])
        accuracy_table.append({
            "Model": model_name,
            "Horizon": "Overall",
            "months_before": -1,
            "RMSE": rmse_all,
            "MAE": mae_all,
            "Bias": bias_all,
            "N_obs": len(model_df),
        })

accuracy_df = pd.DataFrame(accuracy_table)
print("\n" + accuracy_df[["Model", "Horizon", "RMSE", "MAE", "Bias", "N_obs"]].to_string(index=False))

# Highlight best model per horizon
print("\n--- Best Model by Horizon ---")
for m in [3, 2, 1]:
    horizon_df = accuracy_df[(accuracy_df["months_before"] == m)]
    if len(horizon_df) > 0:
        best = horizon_df.loc[horizon_df["RMSE"].idxmin()]
        print(f"  {horizon_labels[m]}: {best['Model']} (RMSE = {best['RMSE']:.4f})")

overall_df = accuracy_df[accuracy_df["Horizon"] == "Overall"]
if len(overall_df) > 0:
    best_overall = overall_df.loc[overall_df["RMSE"].idxmin()]
    print(f"  Overall:        {best_overall['Model']} (RMSE = {best_overall['RMSE']:.4f})")

In [ ]:
# Step 3: Comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

model_colors = {"Bridge": "steelblue", "DFM": "darkorange", "MIDAS": "forestgreen"}

# Top-left: RMSE by horizon grouped by model
ax = axes[0, 0]
horizons = [3, 2, 1]
x = np.arange(len(horizons))
width = 0.25
for i, model_name in enumerate(["Bridge", "DFM", "MIDAS"]):
    model_rmse = []
    for h in horizons:
        row = accuracy_df[(accuracy_df["Model"] == model_name) &
                          (accuracy_df["months_before"] == h)]
        model_rmse.append(row["RMSE"].values[0] if len(row) > 0 else 0)
    ax.bar(x + i * width, model_rmse, width, label=model_name,
           color=model_colors[model_name], edgecolor="black", alpha=0.8)
ax.set_xlabel("Months Before Release")
ax.set_ylabel("RMSE")
ax.set_title("RMSE by Horizon", fontsize=12, fontweight="bold")
ax.set_xticks(x + width)
ax.set_xticklabels([f"{h}m" for h in horizons])
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

# Top-right: Overall RMSE comparison
ax = axes[0, 1]
overall_data = accuracy_df[accuracy_df["Horizon"] == "Overall"]
bars = ax.bar(overall_data["Model"], overall_data["RMSE"],
              color=[model_colors[m] for m in overall_data["Model"]],
              edgecolor="black", alpha=0.8)
ax.set_ylabel("RMSE")
ax.set_title("Overall RMSE", fontsize=12, fontweight="bold")
ax.grid(True, alpha=0.3, axis="y")
for bar, val in zip(bars, overall_data["RMSE"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
            f"{val:.4f}", ha="center", fontsize=10)

# Bottom-left: Nowcast vs Actual scatter by model
ax = axes[1, 0]
for model_name in ["Bridge", "DFM", "MIDAS"]:
    model_df = all_comp_df[all_comp_df["model"] == model_name]
    ax.scatter(model_df["actual"], model_df["nowcast"],
              color=model_colors[model_name], alpha=0.5, s=40, label=model_name)
lims = [all_comp_df[["actual", "nowcast"]].min().min() - 0.5,
        all_comp_df[["actual", "nowcast"]].max().max() + 0.5]
ax.plot(lims, lims, "k--", alpha=0.5, label="Perfect")
ax.set_xlabel("Actual GDP Growth")
ax.set_ylabel("Nowcast")
ax.set_title("Nowcast vs Actual", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Bottom-right: Error boxplot by model
ax = axes[1, 1]
error_data = [all_comp_df[all_comp_df["model"] == m]["error"].values
              for m in ["Bridge", "DFM", "MIDAS"]]
bp = ax.boxplot(error_data, labels=["Bridge", "DFM", "MIDAS"],
                patch_artist=True)
for patch, model_name in zip(bp["boxes"], ["Bridge", "DFM", "MIDAS"]):
    patch.set_facecolor(model_colors[model_name])
    patch.set_alpha(0.7)
ax.axhline(0, color="black", linewidth=1, linestyle="--")
ax.set_ylabel("Nowcast Error")
ax.set_title("Error Distribution by Model", fontsize=12, fontweight="bold")
ax.grid(True, alpha=0.3)

plt.suptitle("Comprehensive Nowcasting Comparison: MIDAS vs Bridge vs DFM",
             fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Step 4: Discussion and recommendations
print("=" * 80)
print("DISCUSSION: When to Use Each Approach")
print("=" * 80)

print("""
1. BRIDGE EQUATIONS
   - Best for: Simple, interpretable nowcasts with few indicators
   - Advantages:
     * Easy to implement and explain to policymakers
     * Fast estimation (OLS)
     * Coefficients have direct economic interpretation
     * Robust with small datasets
   - Limitations:
     * Requires temporal aggregation (loses within-quarter info)
     * Doesn't scale well to many indicators
     * Fill method for missing months is ad-hoc

2. DYNAMIC FACTOR MODEL (DFM)
   - Best for: Large panels of indicators, extracting common dynamics
   - Advantages:
     * Handles many indicators efficiently via dimension reduction
     * Kalman filter naturally handles missing data (ragged edge)
     * News decomposition for interpretability
     * Mixed-frequency support via state-space formulation
   - Limitations:
     * More complex to estimate (EM algorithm)
     * Factor selection (number of factors) is non-trivial
     * Assumes linear factor structure

3. MIDAS REGRESSION
   - Best for: Preserving high-frequency dynamics, few indicators
   - Advantages:
     * Preserves within-quarter (or within-month) information
     * Flexible weight functions (Beta, Almon)
     * Can handle very high frequency ratios (daily to quarterly)
     * Weight function reveals timing of predictive content
   - Limitations:
     * Typically single-equation (one high-freq indicator at a time)
     * NLS estimation can be sensitive to starting values
     * U-MIDAS may overfit with many lags

PRACTICAL RECOMMENDATIONS:
- Start with Bridge for a quick baseline and interpretability
- Use DFM when you have a large panel (>10 indicators)
- Use MIDAS when within-period dynamics matter (financial data)
- Combine forecasts from multiple approaches for robustness
- Always evaluate with pseudo real-time exercises
""")

# Print final summary table
print("\n" + "=" * 80)
print("FINAL SUMMARY TABLE")
print("=" * 80)
summary = accuracy_df[accuracy_df["Horizon"] != "Overall"][
    ["Model", "Horizon", "RMSE", "MAE", "Bias"]
].pivot_table(index="Horizon", columns="Model", values="RMSE")
print("\nRMSE by Horizon and Model:")
print(summary.to_string())

print("\n--- Key Takeaways ---")
print("1. All three methods improve as more data becomes available (lower RMSE at 1m vs 3m)")
print("2. DFM tends to perform best at longer horizons (3m) due to Kalman filtering")
print("3. Bridge equations are competitive at shorter horizons (1m) when most data is available")
print("4. MIDAS captures within-quarter dynamics that bridge equations miss")
print("5. No single method dominates everywhere - combination is recommended")